# Clasificarea boabelor uscate de fasole folosind 3 algoritmi Spark

Acest notebook face parte din proiectul final la disciplina Prelucrarea Volumelor Mari de Date.

Scopul proiectului este construirea unui pipeline simplu si reproductibil in Apache Spark pentru clasificarea tipului de fasole uscata pe baza unor caracteristici numerice.

Algoritmii folosiți sunt: Decision Tree, Random Forest și Logistic Regression

Pas 1: Importuri necesare

In [1]:
import time
import urllib.request

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    RandomForestClassifier,
    LogisticRegression
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

Pas 2: Pornire Spark

In [4]:
spark = (
    SparkSession.builder
    .appName("PVMD_DryBean_RandomForest")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Versiune Spark:", spark.version)

Versiune Spark: 4.1.2


Pas 3: Incarcare set de date

In [15]:
from pyspark.sql.types import DoubleType

# Citim dataset-ul
import os

DATA_PATH = "Dry_Bean_Dataset.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Fișierul {DATA_PATH} nu a fost găsit. "
        "Asigură-te că datasetul se află în același folder cu notebook-ul."
    )

df = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

# NOTĂ: inferSchema poate eșua pe unele versiuni Spark / CSV cu valori mixte.
# Convertim explicit coloanele numerice pentru siguranță.
numeric_cols = [
    "Area",
    "Perimeter",
    "MajorAxisLength",
    "MinorAxisLength",
    "AspectRation",
    "Eccentricity",
    "ConvexArea",
    "EquivDiameter",
    "Extent",
    "Solidity",
    "roundness",
    "Compactness",
    "ShapeFactor1",
    "ShapeFactor2",
    "ShapeFactor3",
    "ShapeFactor4"
]

for col_name in numeric_cols:
    df = df.withColumn(col_name, F.col(col_name).cast(DoubleType()))

# Eliminăm rândurile cu valori NULL generate de cast-ul eșuat
df = df.dropna(subset=numeric_cols + ["Class"])

# Verificăm clasele
print("Distributia claselor:")
df.groupBy("Class").count().show()

print(f"Total rânduri: {df.count()}")
print("Numar total de coloane:", len(df.columns))
df.printSchema()

Distributia claselor:
+--------+-----+
|   Class|count|
+--------+-----+
|    CALI| 1630|
|   SEKER| 2027|
|    SIRA| 2636|
|   HOROZ| 1928|
|  BOMBAY|  522|
|BARBUNYA| 1322|
|DERMASON| 3546|
+--------+-----+

Total rânduri: 13611
Numar total de coloane: 17
root
 |-- Area: double (nullable = true)
 |-- Perimeter: double (nullable = true)
 |-- MajorAxisLength: double (nullable = true)
 |-- MinorAxisLength: double (nullable = true)
 |-- AspectRation: double (nullable = true)
 |-- Eccentricity: double (nullable = true)
 |-- ConvexArea: double (nullable = true)
 |-- EquivDiameter: double (nullable = true)
 |-- Extent: double (nullable = true)
 |-- Solidity: double (nullable = true)
 |-- roundness: double (nullable = true)
 |-- Compactness: double (nullable = true)
 |-- ShapeFactor1: double (nullable = true)
 |-- ShapeFactor2: double (nullable = true)
 |-- ShapeFactor3: double (nullable = true)
 |-- ShapeFactor4: double (nullable = true)
 |-- Class: string (nullable = true)



Pas 4: Alegerea coloanelor de intrare

In [6]:
feature_cols = [c for c in df.columns if c != "Class"]

print("Coloane folosite ca features:")
print(feature_cols)

Coloane folosite ca features:
['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4']


Pas 5: Impartirea datelor train/test

In [7]:
# Împărțim datele în 80% train / 20% test
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Randuri train:", train_df.count())
print("Randuri test:", test_df.count())

Randuri train: 10953
Randuri test: 2658


Pas 6: Definire pipeline comun și algoritmi

In [8]:
label_indexer = StringIndexer(
    inputCol="Class",
    outputCol="label"
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=6,
    seed=42
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=30,
    maxDepth=6,
    seed=42
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50
)

Pas 7: Antrenare si evaluare pentru fiecare algoritm

In [12]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

def train_and_evaluate(model_name, classifier):
    pipeline = Pipeline(stages=[
        label_indexer,
        assembler,
        classifier
    ])

    start_time = time.time()

    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)

    train_time = time.time() - start_time

    accuracy = accuracy_evaluator.evaluate(predictions)
    f1_score = f1_evaluator.evaluate(predictions)

    print("===================================")
    print("Model:", model_name)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-score: {f1_score:.4f}")
    print(f"Timp antrenare: {train_time:.4f} secunde")

    predictions.select("Class", "label", "prediction", "probability") \
        .orderBy(F.rand(seed=42)) \
        .show(5, truncate=False)

    return (
        model_name,
        "Dry Bean Dataset",
        df.count(),
        train_df.count(),
        test_df.count(),
        float(accuracy),
        float(f1_score),
        float(train_time)
    )

pas 8: Tabel benchmark

In [13]:
results_list = []

results_list.append(train_and_evaluate("Decision Tree", dt))
results_list.append(train_and_evaluate("Random Forest", rf))
results_list.append(train_and_evaluate("Logistic Regression", lr))

Model: Decision Tree
Accuracy: 0.9011
F1-score: 0.9008
Timp antrenare: 1.7890 secunde
+--------+-----+----------+---------------------------------------------------------------------------------------------------------------------------------+
|Class   |label|prediction|probability                                                                                                                      |
+--------+-----+----------+---------------------------------------------------------------------------------------------------------------------------------+
|HOROZ   |3.0  |3.0       |[0.0,7.722007722007722E-4,0.0,0.9915057915057915,0.0069498069498069494,7.722007722007722E-4,0.0]                                 |
|SIRA    |1.0  |1.0       |[0.034837235865219876,0.9194745859508852,0.013706453455168474,0.013706453455168474,0.009708737864077669,0.008566533409480296,0.0]|
|BARBUNYA|5.0  |4.0       |[0.0,8.116883116883117E-4,0.0,0.00974025974025974,0.8993506493506493,0.08847402597402597,0.001623

======== Concluzii ========


În acest notebook au fost folosiți trei algoritmi Spark ML pentru clasificarea tipurilor de boabe din datasetul Dry Bean: Decision Tree, Random Forest și Logistic Regression.

Pipeline-ul a inclus citirea datelor, verificarea valorilor lipsă, transformarea etichetei text în label numeric, construirea vectorului de caracteristici, antrenarea modelelor și evaluarea pe setul de test.

Rezultatele finale sunt exprimate prin Accuracy, F1-score și timpul de antrenare. Aceste valori vor fi folosite în raportul final.

Pas 9: Oprire Spark

In [30]:
spark.stop()

======== Instrucțiuni de rulare ========


Pentru rularea proiectului, fișierele `project.ipynb` și `Dry_Bean_Dataset.csv` trebuie să se afle în același director. Notebook-ul poate fi rulat local în Visual Studio Code sau Jupyter Notebook, având instalate Python și PySpark.

Fișierul Dry_Bean_Dataset. csv de poate descărca de aici: https://archive.ics.uci.edu/dataset/602/dry+bean+dataset

Pașii de rulare sunt următorii:

1. Se deschide fișierul `project.ipynb`.
2. Se verifică faptul că fișierul `Dry_Bean_Dataset.csv` se află în același folder cu notebook-ul.
3. Se rulează celulele notebook-ului în ordine, de sus în jos.
4. Prima parte a notebook-ului inițializează sesiunea Spark și încarcă datasetul.
5. Următoarele celule verifică schema datelor, pregătesc caracteristicile, împart datele în set de antrenare și set de test, construiesc pipeline-ul Spark ML și antrenează modelul cele 3 modele: Decision Tree, Random Forest și Logistic Regression.
6. La final sunt afișate metricile de evaluare, inclusiv Accuracy, F1-score și timpul de antrenare.


În folder-ul proiectului este prezent si notebook-ul creat in Databricks pentru rularea în cloud.
Pentru accesare:
1. Click dreapta pe fișier
2. Open with live server

Pentru reproductibilitate, împărțirea datelor a fost realizată folosind `seed=42`, iar datasetul este citit printr-un path relativ:
`Dry_Bean_Dataset.csv`.